---
### **Question 1**
> **Q1: Build Your Personalized Knowledge Base:**
> Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:
> • Take the LAST TWO DIGITS of your roll number. For each digit $d$, compute $\text{category} = [\text{"billing"}, \text{"account"}, \text{"general"}][d \% 3]$. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g. if $d\%3$ gives "account", write a question like “how do I update my registered mobile number”).
> • # Example roll number ...23 -> digits 2, 3
> • # digit 2 -> category[2 % 3] = general
> • # digit 3 -> category[3 % 3] = billing
> **Output:** Print your final 6-row DataFrame.
> 
> ```python
> import pandas as pd
> fixed_entries = [
>  {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
>  "keywords": "fee cost price charge", "category": "billing"},
>  {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
>  "keywords": "password reset login", "category": "account"},
>  {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
>  "keywords": "hours timing open time", "category": "general"},
>  {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
>  "keywords": "pay payment upi fee", "category": "billing"},
> ]
> ```

#### **Roll Number Calculation for 1024170154:**
- **Last Two Digits:** `5` and `4`
- **Digit 5:** `5 % 3 = 2` $\rightarrow$ `category = ["billing", "account", "general"][2]` = **`"general"`**
  - *Question:* `"what is the customer support contact number"`
  - *Answer:* `"You can reach customer support at 1800-123-4567 or support@example.com."`
  - *Keywords:* `"support helpline call contact"`
- **Digit 4:** `4 % 3 = 1` $\rightarrow$ `category = ["billing", "account", "general"][1]` = **`"account"`**
  - *Question:* `"how do i update my registered mobile number"`
  - *Answer:* `"Navigate to Profile > Security > Update Phone Number and verify via OTP."`
  - *Keywords:* `"mobile phone update change"`


In [13]:
import pandas as pd
import numpy as np
import re
import sys

# Solution to Q1:
# 1. 4 Fixed Entries
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    },
]

# 2. 2 Personalized Entries for Roll No 1024170154 (Last digits 5 and 4)
# Digit 5 -> category = "general"
# Digit 4 -> category = "account"
personalized_entries = [
    {
        "question": "what is the customer support contact number",
        "answer": "You can reach customer support at 1800-123-4567 or support@example.com.",
        "keywords": "support helpline call contact",
        "category": "general"
    },
    {
        "question": "how do i update my registered mobile number",
        "answer": "Navigate to Profile > Security > Update Phone Number and verify via OTP.",
        "keywords": "mobile phone update change",
        "category": "account"
    }
]

# Combine into single Knowledge Base
all_entries = fixed_entries + personalized_entries

# Build DataFrame
faq_df = pd.DataFrame(all_entries)

print("="*80)
print(f"Final 6-row Personalized Knowledge Base DataFrame (Roll No: 1024170154)")
print("="*80)
display(faq_df)


Final 6-row Personalized Knowledge Base DataFrame (Roll No: 1024170154)


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,what is the customer support contact number,You can reach customer support at 1800-123-456...,support helpline call contact,general
5,how do i update my registered mobile number,Navigate to Profile > Security > Update Phone ...,mobile phone update change,account


---
### **Question 2**
> **Q2: Generate and Score a Hypothesis**
> Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.


In [14]:
# Solution to Q2:
def extract_clean_tokens(text):
    """Extract alphanumeric words in lowercase."""
    return set(re.findall(r'\b\w+\b', str(text).lower()))

def score_hypothesis(query_str, df):
    """
    Computes confidence score by matching query tokens against FAQ keywords and questions.
    Returns matched entries sorted by confidence in descending order.
    """
    query_tokens = extract_clean_tokens(query_str)
    if not query_tokens:
        return pd.DataFrame()

    results = []
    for idx, row in df.iterrows():
        kw_tokens = extract_clean_tokens(row['keywords'])
        q_tokens = extract_clean_tokens(row['question'])
        
        # Keyword matching has weight 2.0, question token matching has weight 1.0
        kw_matches = query_tokens.intersection(kw_tokens)
        q_matches = query_tokens.intersection(q_tokens)
        
        raw_score = (len(kw_matches) * 2.0) + (len(q_matches) * 1.0)
        max_possible = (len(query_tokens) * 2.0) + (len(query_tokens) * 1.0)
        confidence = round(raw_score / max_possible, 3) if max_possible > 0 else 0.0
        
        if raw_score > 0:
            results.append({
                "Index": idx,
                "Question": row['question'],
                "Answer": row['answer'],
                "Category": row['category'],
                "Confidence": confidence,
                "Raw Score": raw_score,
                "Matched Keywords": list(kw_matches)
            })

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df = results_df.sort_values(by=["Confidence", "Raw Score"], ascending=False).reset_index(drop=True)
    return results_df

# Testing Q2 scoring function
print("Testing Q2 Scoring Function with Sample Queries:")
q1_test = "how to reset password and login"
print(f"\nQuery: '{q1_test}'")
display(score_hypothesis(q1_test, faq_df))

q2_test = "annual fee charge and cost"
print(f"\nQuery: '{q2_test}'")
display(score_hypothesis(q2_test, faq_df))


Testing Q2 Scoring Function with Sample Queries:

Query: 'how to reset password and login'


,Index,Question,Answer,Category,Confidence,Raw Score,Matched Keywords
0,1,how to reset password,Go to Settings > Reset Password.,account,0.556,10.0,"[login, password, reset]"
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.056,1.0,[]
2,5,how do i update my registered mobile number,Navigate to Profile > Security > Update Phone ...,account,0.056,1.0,[]



Query: 'annual fee charge and cost'


,Index,Question,Answer,Category,Confidence,Raw Score,Matched Keywords
0,0,what is the annual fee,The annual fee is Rs 500.,billing,0.533,8.0,"[fee, cost, charge]"
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.200,3.0,[fee]


---
### **Question 3**
> **Q3: Write a function `same_category(category_name, df)` that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.**


In [15]:
# Solution to Q3:
def same_category(category_name, df):
    """
    Returns all questions belonging to the given category.
    """
    cat_clean = category_name.strip().lower()
    subset = df[df['category'].str.lower() == cat_clean]
    return subset[['question', 'answer', 'category']]

# Calling function with personalized entry categories: 'general' and 'account'
print("="*65)
print("Questions belonging to Category: 'general' (Personalized Entry 1)")
print("="*65)
display(same_category("general", faq_df))

print("\n" + "="*65)
print("Questions belonging to Category: 'account' (Personalized Entry 2)")
print("="*65)
display(same_category("account", faq_df))


Questions belonging to Category: 'general' (Personalized Entry 1)


,question,answer,category
2,what are your working hours,We are open 9 AM to 5 PM.,general
4,what is the customer support contact number,You can reach customer support at 1800-123-456...,general



Questions belonging to Category: 'account' (Personalized Entry 2)


,question,answer,category
1,how to reset password,Go to Settings > Reset Password.,account
5,how do i update my registered mobile number,Navigate to Profile > Security > Update Phone ...,account


---
### **Question 4**
> **Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named `<your_roll_number>_faq_data.csv`.**


In [16]:
# Solution to Q4:
# Pick entry index 0
entry_idx = 0
print(f"Selected Entry [{entry_idx}] Before Update:")
print("Question:", faq_df.loc[entry_idx, 'question'])
print("Current Keywords:", faq_df.loc[entry_idx, 'keywords'])

# User input with automated fallback default
new_kw = "subscription"
try:
    if hasattr(sys.stdin, 'isatty') and sys.stdin.isatty():
        user_input = input(f"Enter new keyword to add to entry {entry_idx} (default '{new_kw}'): ").strip()
        if user_input:
            new_kw = user_input
except Exception:
    pass

# Add new keyword to the entry's keywords
faq_df.loc[entry_idx, 'keywords'] = faq_df.loc[entry_idx, 'keywords'] + " " + new_kw
print(f"\nUpdated Keywords for Entry [{entry_idx}]:", faq_df.loc[entry_idx, 'keywords'])

# Save to CSV named <your_roll_number>_faq_data.csv
csv_filename = f"1024170154_faq_data.csv"
faq_df.to_csv(csv_filename, index=False)
print(f"\nSuccessfully exported updated Knowledge Base to '{csv_filename}'!")

# Verification: Read back the saved CSV file
loaded_df = pd.read_csv(csv_filename)
print("\nVerification - DataFrame reloaded from CSV:")
display(loaded_df)


Selected Entry [0] Before Update:
Question: what is the annual fee
Current Keywords: fee cost price charge

Updated Keywords for Entry [0]: fee cost price charge subscription

Successfully exported updated Knowledge Base to '1024170154_faq_data.csv'!

Verification - DataFrame reloaded from CSV:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge subscription,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,what is the customer support contact number,You can reach customer support at 1800-123-456...,support helpline call contact,general
5,how do i update my registered mobile number,Navigate to Profile > Security > Update Phone ...,mobile phone update change,account


---
### **Question 5**
> **Q5: Using groupby, print how many FAQ entries you have per category.**


In [17]:
# Solution to Q5:
# Using groupby to count FAQ entries per category
category_counts = faq_df.groupby('category').size().reset_index(name='FAQ Count')

print("="*45)
print("FAQ Entries Count per Category:")
print("="*45)
display(category_counts)

print("\nCategory Breakdown:")
for _, row in category_counts.iterrows():
    print(f"  • {row['category'].capitalize():<12} : {row['FAQ Count']} FAQ entries")


FAQ Entries Count per Category:


,category,FAQ Count
0,account,2
1,billing,2
2,general,2



Category Breakdown:
  • Account      : 2 FAQ entries
  • Billing      : 2 FAQ entries
  • General      : 2 FAQ entries


---
### **Question 6**
> **Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.**


In [18]:
# Solution to Q6:
def score_hypothesis_tie_aware(query_str, df):
    """
    Enhanced scoring function that detects and displays ties for the highest score.
    """
    results = score_hypothesis(query_str, df)
    
    print("="*75)
    print(f"QUERY: '{query_str}'")
    print("="*75)
    
    if results.empty:
        print("No matching FAQ entries found.")
        return results

    top_confidence = results['Confidence'].iloc[0]
    top_matches = results[results['Confidence'] == top_confidence]
    
    if len(top_matches) > 1:
        print(f"\n🔔 TIE DETECTED! Found {len(top_matches)} entries with identical highest confidence ({top_confidence}):")
        for i, (_, row) in enumerate(top_matches.iterrows(), 1):
            print(f"\n  [Candidate Match {i}] - Category: [{row['Category'].upper()}]")
            print(f"    • Question: {row['Question']}")
            print(f"    • Answer:   {row['Answer']}")
            print(f"    • Matched Keywords: {row['Matched Keywords']}")
    else:
        best_match = top_matches.iloc[0]
        print(f"\n🎯 UNIQUE BEST MATCH (Confidence: {top_confidence}):")
        print(f"    • Category: [{best_match['Category'].upper()}]")
        print(f"    • Question: {best_match['Question']}")
        print(f"    • Answer:   {best_match['Answer']}")
        print(f"    • Matched Keywords: {best_match['Matched Keywords']}")

    print("\nAll Ranked Hypothesis Results:")
    display(results)
    return results

# Demonstration 1: Query that PRODUCES A TIE (both billing entries match 'fee')
print("DEMONSTRATION 1: QUERY PRODUCING A TIE")
query_tie = "what is the fee"
score_hypothesis_tie_aware(query_tie, faq_df)

print("\n" + "#"*80 + "\n")

# Demonstration 2: Query that DOES NOT PRODUCE A TIE
print("DEMONSTRATION 2: QUERY WITHOUT A TIE")
query_no_tie = "how to reset password"
score_hypothesis_tie_aware(query_no_tie, faq_df)


DEMONSTRATION 1: QUERY PRODUCING A TIE
QUERY: 'what is the fee'

🎯 UNIQUE BEST MATCH (Confidence: 0.5):
    • Category: [BILLING]
    • Question: what is the annual fee
    • Answer:   The annual fee is Rs 500.
    • Matched Keywords: ['fee']

All Ranked Hypothesis Results:


,Index,Question,Answer,Category,Confidence,Raw Score,Matched Keywords
0,0,what is the annual fee,The annual fee is Rs 500.,billing,0.500,6.0,[fee]
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.333,4.0,[fee]
2,4,what is the customer support contact number,You can reach customer support at 1800-123-456...,general,0.250,3.0,[]
3,2,what are your working hours,We are open 9 AM to 5 PM.,general,0.083,1.0,[]



################################################################################

DEMONSTRATION 2: QUERY WITHOUT A TIE
QUERY: 'how to reset password'

🎯 UNIQUE BEST MATCH (Confidence: 0.667):
    • Category: [ACCOUNT]
    • Question: how to reset password
    • Answer:   Go to Settings > Reset Password.
    • Matched Keywords: ['password', 'reset']

All Ranked Hypothesis Results:


,Index,Question,Answer,Category,Confidence,Raw Score,Matched Keywords
0,1,how to reset password,Go to Settings > Reset Password.,account,0.667,8.0,"[password, reset]"
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.083,1.0,[]
2,5,how do i update my registered mobile number,Navigate to Profile > Security > Update Phone ...,account,0.083,1.0,[]


,Index,Question,Answer,Category,Confidence,Raw Score,Matched Keywords
0,1,how to reset password,Go to Settings > Reset Password.,account,0.667,8.0,"[password, reset]"
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.083,1.0,[]
2,5,how do i update my registered mobile number,Navigate to Profile > Security > Update Phone ...,account,0.083,1.0,[]
